### DESCRIPTION (inspired by Leetcode.com)
medium

A travel booking system needs to find the lowest-cost route between airports. You're given n cities connected by flights, where each flight [from, to, price] represents a direct route with its cost.

Find the cheapest route from src to dst that uses at most k layovers (intermediate cities). If no such route exists, return -1.

Note: A route with k layovers means visiting k intermediate cities, or equivalently, taking k+1 flights.

Example 1:

Input:
```
n = 4
flights = [[0,1,100], [1,2,100], [2,3,100], [0,3,500]]
src = 0
dst = 3
k = 1
```
Output: 500


Explanation: The route 0→1→2→3 costs $300 with 2 layovers (cities 1 and 2), which exceeds k=1. The direct flight 0→3 costs $500 with 0 layovers. Even though it's more expensive, it's the only valid route.

Example 2:

Input:
```
n = 4
flights = [[0,1,100], [1,2,100], [2,3,100]]
src = 0
dst = 3
k = 1
```
Output: -1

Explanation: The only path 0→1→2→3 requires 2 layovers (cities 1 and 2), but we're limited to k=1. No valid route exists.

In [16]:
from typing import List
from collections import defaultdict
import heapq

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # 1. create adjlist
        adjlist = defaultdict(list)
        for u, v, w in flights:
            adjlist[u].append((v, w))

        # 2. create dictionary to keep track of lowest cost from source for each node and min-heap for cost, # of stops from source to current visiting node.
        cheapest = {src: 0}
        heap = [[0, 0, src]]

        # 3. while heap is not empty, pop each node.
        # for each of node's neighbor, calculate new cost (cost from source + current cost to neighbor) and # of stops. 
        # If new cost is less than what we have in dictionary and stops is same or less than k, update the dictionary
        while heap:
            stops, cost_from_src, node = heapq.heappop(heap)
            for neighbor, cost in adjlist[node]:
                new_cost = cost_from_src + cost
                if new_cost < cheapest.get(neighbor, float('inf')) and stops <= k:
                    cheapest[neighbor] = new_cost
                    heapq.heappush(heap, [stops+1, new_cost, neighbor])

        # 4. return the cost to dst in dictionary
        result = cheapest.get(dst, float('inf'))
        return result if result < float('inf') else -1

### Feedback
Your approach passes the tests and has the right general bounded-search idea, but the heap tuple is ordered as (stops, cost, node), so this is not truly cost-prioritized. More importantly, cheapest[node] stores only the cheapest cost regardless of stops. A slightly more expensive route to a city may use fewer flights and enable a cheaper valid route later, while your pruning would discard it. For example, reaching the same city cheaply with too many flights can block a valid route with fewer flights. Track the best cost by (city, flights_used) (or maintain costs per stop count). Also, your variable stops actually counts flights used: starting at 0 and pushing stops + 1; the condition stops <= k correctly permits at most k + 1 flights, but clearer naming would reduce off-by-one confusion. Complexity can be stated using the bounded state space.


In [ ]:
from typing import List
from collections import defaultdict
import heapq

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # 1. create adjlist
        adjlist = defaultdict(list)
        for u, v, w in flights:
            adjlist[u].append((v, w))

        # 2. create dictionary to keep track of lowest cost from source for each node and the # of stops. 
        # Create min-heap for cost, node and # of stops. This order is important. We are prioritizing cost first (cheapest), and then destination. The number of stops should be last.
        cheapest = {(src,0): 0}
        heap = [[0, src, 0]]

        # 3. while heap is not empty, pop each node.
        # if node is the destination, return the cost. We can short cut here because min-heap quarantees we reach the cheapest combination first.
        # if stops is more than k, can skip since we won't consider this route.
        # for each of node's neighbor, calculate new cost (cost from source + current cost to neighbor) and # of stops. 
        # If new cost is less than what we have in dictionary, update the dictionary
        while heap:
            cost_from_src, node, stops = heapq.heappop(heap)
            if node == dst: 
                return cost_from_src
            if stops > k:
                continue

            for neighbor, cost in adjlist[node]:
                new_cost = cost_from_src + cost
                if new_cost < cheapest.get((neighbor, stops+1), float('inf')):
                    cheapest[(neighbor, stops+1)] = new_cost
                    heapq.heappush(heap, [new_cost, neighbor, stops+1])

        return -1

### Feedback

Your solution is correct and passed all tests. The heap ordering makes early return from dst safe: every queued route is ordered by cost, and routes exceeding k flights are never expanded. Your state key (node, stops) correctly avoids conflating paths with different stop counts.

One small clarity issue: stops actually counts flights taken, not layovers. Starting at 0 and allowing expansion while stops <= k correctly permits k + 1 flights, but naming/commenting this explicitly would prevent off-by-one confusion in an interview.

You could also discard stale heap entries when a popped cost is no longer the recorded minimum for that state, though your per-state relaxation already keeps the algorithm efficient. Complexity is roughly O(Ek log(Ek)) time and O(Ek) space.



In [2]:
from typing import Callable

class Input:
    def __init__(self, n:int, flights: List[List[int]], src: int, dst:int, k: int):
        self.flights = flights
        self.n = n
        self.k = k
        self.src = src
        self.dst = dst
        
class Test:  
    def __init__(self, input: Input, result: int):
        self.input = input
        self.expected_result = result
        
def run_tests(tests: list[Test], func: Callable[[int, List[List[int]], int, int, int], int]):
    for test in tests:
        result = func(test.input.n, test.input.flights, test.input.src, test.input.dst, test.input.k)
        if result == test.expected_result:
            print(f"Test passed for {test.input.flights}")
        else:
            print(f"Test failed for {test.input.flights}. Expected: {test.expected_result}, Actual: {result}")

In [22]:
tests = [
    Test(Input(2, [[0,1,100]], 0, 1, 1), 100),
    Test(Input(2, [[0,1,200]], 0, 1, 0), 200),
    Test(Input(3, [[0,1,100], [1,2,100], [0,2,300]], 0, 2, 1), 200),
    Test(Input(4, [[0,1,100], [1,2,100], [2,3,100], [0,3,500]], 0, 3, 1), 500),
    Test(Input(4, [[0,1,1000], [1,2,1000], [2,3,1000]], 0, 3, 1), -1),
    Test(Input(5, [[0,1,1],[1,2,1],[0,2,10],[2,3,1],[3,4,1]], 0, 4, 2), 12)
    
]

run_tests(tests, Solution().findCheapestPrice)

Test passed for [[0, 1, 100]]
Test passed for [[0, 1, 200]]
Test passed for [[0, 1, 100], [1, 2, 100], [0, 2, 300]]
Test passed for [[0, 1, 100], [1, 2, 100], [2, 3, 100], [0, 3, 500]]
Test passed for [[0, 1, 1000], [1, 2, 1000], [2, 3, 1000]]
Test passed for [[0, 1, 1], [1, 2, 1], [0, 2, 10], [2, 3, 1], [3, 4, 1]]
